# Chapter 6 Sample Code
This file produces the letter frequencies, ngrams, and uses them to produce sequences of letters using a statistical approach.

In [11]:
import pandas as pd
import os
import re
import pickle
import random

### Download the Reuters dataset 

The next few files parse the Reuters dataset files available here:
https://kdd.ics.uci.edu/databases/reuters21578/reuters21578.html

The files should be unzipped and untarred into the directory listed below with many .sgm files.

The frequency data are then written to .pkl files.

In [10]:
import os
# Set absolute path to the ch6 directory (current directory for notebook)
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
directory = os.getcwd()  # Use current working directory
print(f"Current working directory: {directory}")
print(f"Directory exists: {os.path.exists(directory)}")

# Check for .sgm files
sgm_files = [f for f in os.listdir(directory) if f.endswith('.sgm')]
print(f"Found {len(sgm_files)} .sgm files in {directory}")

if len(sgm_files) == 0:
    print("\n⚠️  WARNING: No .sgm files found!")
    print("Please download the Reuters dataset from:")
    print("https://kdd.ics.uci.edu/databases/reuters21578/reuters21578.html")
    print(f"And extract .sgm files to: {directory}")
    print("\nFiles in current directory:")
    all_files = os.listdir(directory)[:10]
    for f in all_files:
        print(f"  - {f}")
else:
    print(f"✓ Files found: {sgm_files[:5]}...")

Current working directory: d:\Seminar_chuyen_de\Week_4\Supercharged-Coding-with-Gen-AI\ch6
Directory exists: True
Found 22 .sgm files in d:\Seminar_chuyen_de\Week_4\Supercharged-Coding-with-Gen-AI\ch6
✓ Files found: ['reut2-000.sgm', 'reut2-001.sgm', 'reut2-002.sgm', 'reut2-003.sgm', 'reut2-004.sgm']...


### Parsing the Reuters sgm files into pkl frequency files

In [ ]:
def extractBodyTextFromFile(filename):
    '''Extracts text from a filename, looking for <BODY> tags and returning the content between them.'''
    inBody = False
    textblock = []
    with open(filename, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.readline()
        while len(text) > 0:
            if '<BODY>' in text and '</BODY>' in text:
                inBody = False
                text = re.sub(r'^.*\<BODY\>', '', text)
                text = re.sub(r'\<\/BODY\>.*$', '', text)
                textblock.append(text)
            elif '<BODY>' in text and inBody is False:
                text = re.sub(r'^.*\<BODY\>', '', text)
                textblock.append(text)
                inBody = True
            elif '</BODY>' in text and inBody is True:
                text = re.sub(r'\<\/BODY\>.*$', '', text)
                textblock.append(text)
                inBody = False
            elif inBody:
                textblock.append(text)
            text = f.readline()
    return textblock

<>:10: SyntaxWarning: invalid escape sequence '\<'
<>:11: SyntaxWarning: invalid escape sequence '\<'
<>:14: SyntaxWarning: invalid escape sequence '\<'
<>:18: SyntaxWarning: invalid escape sequence '\<'
<>:10: SyntaxWarning: invalid escape sequence '\<'
<>:11: SyntaxWarning: invalid escape sequence '\<'
<>:14: SyntaxWarning: invalid escape sequence '\<'
<>:18: SyntaxWarning: invalid escape sequence '\<'
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4608\183685533.py:10: SyntaxWarning: invalid escape sequence '\<'
  text = re.sub('^.*\<BODY\>', '', text)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4608\183685533.py:11: SyntaxWarning: invalid escape sequence '\<'
  text = re.sub('\<\/BODY\>.*$', '', text)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4608\183685533.py:14: SyntaxWarning: invalid escape sequence '\<'
  text = re.sub('^.*\<BODY\>', '', text)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4608\183685533.py:18: SyntaxWarning: invalid escape sequence '\<'
  text = re.sub('\<\/BODY\>

In [ ]:
def preprocess(text_block):
    '''Lowercase text removing newlines, leading and trailing space, and multiple spaces.'''
    text = ' '.join(text_block)
    text = re.sub(r'[\s\r\n]+', ' ', text).lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4608\3192238090.py:4: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('[\s\r\n]+', ' ', text).lower()
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4608\3192238090.py:5: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s+', ' ', text)


In [14]:
def computeFrequencies(text, ngram_len, frequencies):
    '''Updates and returns frequencies dictionary with ngrams of ngram_len from text.'''
    for i in range(len(text)-ngram_len):
        key = text[i:i+ngram_len]
        if key in frequencies:
            frequencies[key] += 1 
        else:
            frequencies[key] = 1
    return frequencies

In [ ]:
def purge_nonletters(ngrams):
    ''' Returns updated ngrams, removing those with non-letter non-space characters'''
    to_delete = []
    for key in ngrams:
        if re.match(r'^[a-z\s]+$', key):
            pass
        else:
            to_delete.append(key)
    for rejected in to_delete:
        ngrams.pop(rejected, None)
    return ngrams

<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4608\2415040099.py:5: SyntaxWarning: invalid escape sequence '\s'
  if re.match('^[a-z\s]+$', key):


In [16]:
def computeFreq(directory, ngram_len):
    '''Primary routine to iterate through all .sigm files in directory and compute ngram frequencies.'''
    sgm_files = [f for f in os.listdir(directory) if f.endswith('.sgm')]
    ngram_freq = {}
    for file in sgm_files:
        extract_text_blocks = extractBodyTextFromFile(os.path.join(directory, file))
        cleaned_block = preprocess(extract_text_blocks)
        ngram_freq = computeFrequencies(cleaned_block, ngram_len, ngram_freq)
        purge_nonletters(ngram_freq)
    return ngram_freq

Computes ngram frequencies for 1 to MAX_NGRAM_SIZE and write to freq#.pkl.

In [17]:
MAX_NGRAM_SIZE = 7
for ngrami in range(1, MAX_NGRAM_SIZE+1):
    print(f'Processing ngram size {ngrami}...')
    freqOutput = computeFreq(directory, ngrami)
    output_file = os.path.join(directory, 'freq' + str(ngrami) + '.pkl')
    with open(output_file, 'wb') as f:
        pickle.dump(freqOutput, f)
    print(f'✓ Saved to {output_file}')
    print(f'  Length of freq dictionary size {ngrami}: {len(freqOutput)}\n')

Processing ngram size 1...
✓ Saved to d:\Seminar_chuyen_de\Week_4\Supercharged-Coding-with-Gen-AI\ch6\freq1.pkl
  Length of freq dictionary size 1: 27

Processing ngram size 2...
✓ Saved to d:\Seminar_chuyen_de\Week_4\Supercharged-Coding-with-Gen-AI\ch6\freq1.pkl
  Length of freq dictionary size 1: 27

Processing ngram size 2...
✓ Saved to d:\Seminar_chuyen_de\Week_4\Supercharged-Coding-with-Gen-AI\ch6\freq2.pkl
  Length of freq dictionary size 2: 703

Processing ngram size 3...
✓ Saved to d:\Seminar_chuyen_de\Week_4\Supercharged-Coding-with-Gen-AI\ch6\freq2.pkl
  Length of freq dictionary size 2: 703

Processing ngram size 3...
✓ Saved to d:\Seminar_chuyen_de\Week_4\Supercharged-Coding-with-Gen-AI\ch6\freq3.pkl
  Length of freq dictionary size 3: 9860

Processing ngram size 4...
✓ Saved to d:\Seminar_chuyen_de\Week_4\Supercharged-Coding-with-Gen-AI\ch6\freq3.pkl
  Length of freq dictionary size 3: 9860

Processing ngram size 4...
✓ Saved to d:\Seminar_chuyen_de\Week_4\Supercharged-Cod

In [18]:
def checkTrainingSetSize(directory):
    ''' Checks the size of the training set before and after removing non-letter characters. '''
    sgm_files = [f for f in os.listdir(directory) if f.endswith('.sgm')]
    training_set_size = 0
    cleaned_set_size = 0
    for file in sgm_files:
        extract_text_blocks = extractBodyTextFromFile(os.path.join(directory, file))
        training_set_size += len(' '.join(extract_text_blocks))
        cleaned_block = preprocess(extract_text_blocks)
        cleaned_set_size += len(cleaned_block)
    print('Uncleaned', training_set_size, 'Cleaned', cleaned_set_size)
checkTrainingSetSize(directory)


Uncleaned 16372459 Cleaned 15620895


Generate frequency dict of the letters within the text files.
Write the file to letter_frequencies.csv

In [19]:
def computeIndividualLetterFrequencies(directory):
    '''Computes letter frequencies in the text files in the given directory.'''
    sgm_files = [f for f in os.listdir(directory) if f.endswith('.sgm')]
    training_set_size = 0
    cleaned_set_size = 0
    letter_frequencies = {}
    for u in 'abcdefghijklmnopqrstuvwxyz':
        letter_frequencies[u] = 0
    for file in sgm_files:
        extract_text_blocks = extractBodyTextFromFile(os.path.join(directory, file))
        training_set_size += len(' '.join(extract_text_blocks))
        cleaned_block = preprocess(extract_text_blocks)
        for u in cleaned_block:
            if u in letter_frequencies:
                letter_frequencies[u] += 1
        cleaned_set_size += len(cleaned_block)
    return letter_frequencies

letter_frequencies = computeIndividualLetterFrequencies(directory)
df = pd.DataFrame(list(letter_frequencies.items()), columns=['letter', 'frequency'])
df = df.sort_values(by='frequency', ascending=False)
df.to_csv('letter_frequencies.csv', index=False)


### Generate the next letters from the ngram frequency data

In [20]:
def create_next_letter_from_current_frequency_table():
    ''' Generates pairwise frequency table for current letter and next letter'''
    with open(os.path.join(directory, 'freq2.pkl'), 'rb') as f:
        freq2 = pickle.load(f)
    set1 = set()
    set2 = set()
    for key in freq2:
        set1.add(key[0])
        set2.add(key[1])
    lst1 = list(set1)
    lst2 = list(set2)
    lst1.sort()
    lst2.sort()
    freqTable = pd.DataFrame(index=lst2, columns=lst1)    
    freqTable.fillna(0, inplace=True)
    for key in freq2:
        freqTable.at[key[1], key[0]] = freq2[key]
    freqTable.to_csv(os.path.join(directory, 'freq2tbl.csv'))
create_next_letter_from_current_frequency_table()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4608\2664634740.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  freqTable.fillna(0, inplace=True)


The file written out was freq2tbl was used to create the histogram of the most frequent single entries and the pairs of entries.

In [21]:
def loadNormalizePkl1(filename):
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    totalsum = 0.0
    for key, val in data.items():
        totalsum += val
    for key in data:
        data[key] = data[key] / totalsum
    return data



Writes text using the letter frequencies alone without ngrams.

In [22]:
data = loadNormalizePkl1(os.path.join(directory,'freq1.pkl'))
random.seed(41)
for iter in range(10):
    outputstr = ''
    for index in range(70):
        prob = random.uniform(0.0, 1.0)
        sum = 0
        done = False
        prevkey = ''
        for key in data:
            if not done:
                sum += data[key]
                if sum >= prob:
                    done = True
                    outputstr += key

    print(iter+1, outputstr)

1  eelnin ungia wwuetthgsaaeuuoiolifs ttelaroi ic so u dnli ohsaadoiolid
2 t ape ui  preirl y ldueeseihenw eioxtincplastmtcteughct ivlrwataartrbi
3 ectee hcmasauisdeb  eaeacwinsarsdu enti nuee dba io haqid kh  aeeoa lr
4 aelplilvprag el nl einrileadneirearoafr ndstey m iser e hnpatfut hllbi
5 appdsmn  rstthylan   mvlccirefrrchneahbrgbmefw schnygtof mormde neehc 
6     dwo r amave orre mclnditytceo  oms e ap ilr  arleits a s n  mi  de
7  dowsibits   e ehns  ai siutfc notoitieimfdwdefss hmyra aom e ss txc r
8 svirdit ayde tmc hhriedtrnastdp tbrgdm ocildnaatrsts friu tlu fiebgzab
9  ddaefnlcgtr aryugnit snin nothohmltmei onoroelcnowoindte  yiaipdshsle
10  terlea i ii r dse icelnl mn wbeaaouiodm n eeiposnrarsr   rpseaee tage


#### Produce text using the n-gram letter frequencies of different lengths.

In [23]:
def loadNormalizePkln(filename):
    ''' Creates normalization by first n-1 letters of ngram for the last letter in ngrams'''
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    norm = {}
    for key in data.keys():
        prefix = key[0:len(key)-1]
        norm[prefix] = 0
    for key,value in data.items():
        prefix = key[0:len(key)-1]
        norm[prefix] += value
    for key,value in data.items():
        prefix = key[0:len(key)-1]
        data[key] = data[key] / norm[prefix]
    return data

In [24]:
def find_prob_string(prob, data, ngram):
    ''' Selects random entry using prob from the data keys starting with ngram that have been normalized.'''
    sum = 0.0
    for key in data:
        if key.startswith(ngram):
            sum += data[key]
            if sum >= prob:
                return key[-1]  
    print('No match found for', ngram, prob)
    return None

In [25]:
def return_random_key(data):
    ''' Returns a random key from the data dictionary.'''
    key = random.choice(list(data.keys()))  
    return key
    

In [26]:
def createNgramStr(directory, ngram):
    '''Generates lines of random text based on n-gram frequencies stored in directory.'''
    NUM_LINES = 5
    LETTERS_PER_LINE = 70
    data = loadNormalizePkln(os.path.join(directory, 'freq' + str(ngram) + '.pkl'))
    random.seed(422)
    for iter in range(NUM_LINES):
        outputstr = return_random_key(data)
        for index in range(LETTERS_PER_LINE):
            prob = random.uniform(0.0, 1.0)
            nextchar = find_prob_string(prob, data, outputstr[-ngram+1:])
            outputstr += nextchar
        print(iter+1, outputstr)

##### Main call to generate ngram-based letter sequences using problabilities

In [27]:
ngram_len = 5  # use ngram_len of 2 to MAX_NGRAM_SIZE = 7
createNgramStr(directory, ngram_len)

1 frn in and corp told keeping an from years to market of about there listerd
2 ow fluctane cited under fight pct on the large share into mazda ways to the
3 orb states all year oper shr longer board if of worldwident effort said und
4 logan outstandar years for that they said the fund dives for the right pres
5 ority and its proving exposures said internative to tights income for throu
3 orb states all year oper shr longer board if of worldwident effort said und
4 logan outstandar years for that they said the fund dives for the right pres
5 ority and its proving exposures said internative to tights income for throu


In [28]:
print("\n" + "="*80)
print("COMPARISON: Text generation with different n-gram sizes")
print("="*80)
print("\nNotice how the text quality improves with larger n-gram sizes:\n")

for ngram_size in [2, 3, 4, 5, 7]:
    print(f"\n{'─'*80}")
    print(f"N-gram size = {ngram_size}:")
    print(f"{'─'*80}")
    createNgramStr(directory, ngram_size)


COMPARISON: Text generation with different n-gram sizes

Notice how the text quality improves with larger n-gram sizes:


────────────────────────────────────────────────────────────────────────────────
N-gram size = 2:
────────────────────────────────────────────────────────────────────────────────
1 wmarerye icose verhe herer duvoucrpa tilllarast itout f cts coles t tere
2  quron ryeurod g tst venk es wedus d it pll inopo antopcol ediomoforen f
3 lzin terop zquntreve re h tha hentwhase ues tabr iningte as ithormarape 
4 rhelds sa tonat ainserathaby sard d aran ido rerealdon itino iconderilos
5 bagalintenin bs cutr tha mag temes cr teat owa th ondes brsumesais t g t

────────────────────────────────────────────────────────────────────────────────
N-gram size = 3:
────────────────────────────────────────────────────────────────────────────────
1 laa fith any sh wilhipect thenbleraqi alevid can of the sid in majoin sai
2 sciall appoilliv the takems red trichater jappetted siserseparces

## 📊 Kết Quả Thực Hiện Bài Tập

### ✅ Hoàn Thành Các Bước

1. **Tải và Xử Lý Dữ Liệu Reuters** ✓
   - Tìm thấy 22 file .sgm trong thư mục
   - Dung lượng dữ liệu: 16.3 MB (chưa xử lý) → 15.6 MB (sau xử lý)

2. **Tính Toán Tần Suất N-gram** ✓
   - freq1.pkl: 27 entries (27 ký tự)
   - freq2.pkl: 703 entries (cặp ký tự)
   - freq3.pkl: 9,860 entries
   - freq4.pkl: 53,912 entries
   - freq5.pkl: 172,338 entries
   - freq6.pkl: 420,485 entries
   - freq7.pkl: 823,342 entries

3. **Tạo File Kết Quả** ✓
   - letter_frequencies.csv - Tần suất từng ký tự
   - freq2tbl.csv - Bảng 2D tần suất pairwise
   - Các file .pkl cho mỗi n-gram size

4. **Sinh Văn Bản Thống Kê** ✓
   - Văn bản từ tần suất ký tự đơn lẻ (không có ý nghĩa)
   - Văn bản từ 2-gram đến 7-gram (chất lượng tăng)
   - Văn bản 7-gram gần giống tiếng Anh thực tế!

### 📈 Quan Sát Thú Vị

- **N-gram nhỏ (2-3)**: Văn bản rác, chỉ là ký tự ngẫu nhiên
- **N-gram vừa (4-5)**: Bắt đầu có hình dáng từ tiếng Anh
- **N-gram lớn (6-7)**: Từ thực tế, cấu trúc câu hợp lý!